# Diffusion stages: DiT, CFG, latents, and VAE

## Learning goals
- Run a denoising loop over latents and watch it converge
- See classifier-free guidance change the trajectory
- Map latent shapes to pixel shapes through a VAE

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## Diffusion is not autoregressive

A DiT stage starts from pure noise in a compressed **latent** space and iteratively denoises toward a sample. There is no KV cache and no token-by-token loop — instead, a fixed number of denoising steps each run a full forward pass. A **VAE** then decodes the final latent to pixels.

```text
noise latent -> [DiT x N steps] -> clean latent -> [VAE decode] -> image
```

In [ ]:
from omni_tutorial import denoise, distance_to_target, vae_decode_shape
import numpy as np
rng = np.random.default_rng(1)
latent = rng.standard_normal((8, 8, 4))   # (h, w, channels) in latent space
target = rng.standard_normal((8, 8, 4))   # the sample the toy 'model' pulls toward
traj = denoise(latent, target, steps=25, cfg_scale=1.0)
print(f'{len(traj)-1} denoising steps; final distance to target '
      f'{distance_to_target(traj, target)[-1]:.3f}')

## Classifier-free guidance

CFG combines a conditional and unconditional prediction: `guided = uncond + scale * (cond - uncond)`. A scale above 1 extrapolates past the conditional prediction, pushing harder toward the prompt. We sweep the scale and plot how fast each run approaches the target.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from omni_tutorial import denoise, distance_to_target
fig, ax = plt.subplots(figsize=(6, 3.4))
for scale in (1.0, 2.0, 4.0):
    traj = denoise(latent, target, steps=25, cfg_scale=scale)
    ax.plot(distance_to_target(traj, target), label=f'cfg={scale}')
ax.set_xlabel('denoising step'); ax.set_ylabel('L2 distance to target')
ax.set_title('Guidance strength vs convergence'); ax.legend()
fig.tight_layout(); plt.show()

## Latent -> pixel through the VAE

The DiT works in a spatially compressed latent (typically 8x smaller per axis). The VAE decoder expands it back to RGB pixels.

In [ ]:
for latent_shape in [(8, 8, 4), (64, 64, 4), (96, 160, 4)]:
    print(f'latent {latent_shape} -> image {vae_decode_shape(latent_shape)}')

## Exercise

At very high CFG the curve can overshoot and rise again. Predict at which scale, then sweep `cfg_scale` in the plot cell to find where convergence stops improving.

In [ ]:
# Solution: extrapolation past the target eventually overshoots.
for scale in (1, 2, 4, 8):
    final = distance_to_target(denoise(latent, target, steps=25, cfg_scale=scale), target)[-1]
    print(f'cfg={scale}: final distance {final:.3f}')

## Source lab

Read a diffusion model's deploy config in `vllm_omni/deploy/` and find its scheduler / step count and where the VAE decode happens. Compare to the AR configs from nb03.